# TD 4 · Diagnostiquer, dimensionner, comparer honnêtement

**Bloc 3 : deep learning · Séance 9 · Durée : 80 min · Niveau : ★★☆ à ★★★**

---

## Objectifs

À la fin de ce TD, vous serez capable de :

1. diagnostiquer un entraînement à partir de ses seules courbes, et choisir l'action ;
2. trouver un bug silencieux avec une checklist, dont le **test des 20 lignes**, et
   connaître les limites de ce test ;
3. mesurer l'effet de la capacité et situer le plancher et le plateau ;
4. montrer que la profondeur n'aide que si la machinerie suit, en mesurant le
   vanishing gradient ;
5. mener une comparaison honnête contre le champion de la séance 7 et rédiger une
   conclusion bornée.

## Prérequis

- TD 3 (pipeline, training loop, `fit`) ;
- séance 6 (paired t-test, lecture des signes), séance 7 (HistGradientBoosting, AUC
  0,9216) ;
- cours : partie 2 du diaporama (diagnostic, capacité, comparaison honnête).

## Concepts mobilisés

underfitting, overfitting, entraînement cassé · checklist de débogage · test des 20
lignes · capacité (largeur, nombre de paramètres) · profondeur · vanishing gradient ·
cross-validation groupée · paired t-test · portée d'une conclusion.

## Déroulé

| Exercice | Contenu                                              | Durée  | Type         | Niveau |
| -------- | ---------------------------------------------------- | ------ | ------------ | ------ |
| 4.1      | Lire des courbes à l'aveugle                         | 12 min | analyser     | ★★☆    |
| 4.2      | Cinq training loops mystère et le test des 20 lignes | 15 min | résoudre     | ★★★    |
| 4.3      | Capacité : plancher et plateau                       | 13 min | expérimenter | ★★☆    |
| 4.4      | Profondeur et vanishing gradient                     | 15 min | comparer     | ★★☆    |
| 4.5      | La comparaison honnête                               | 20 min | résoudre     | ★★★    |
|          | Synthèse et questions                                | 5 min  |              |        |

## Résultats attendus

- Quatre courbes correctement classées (cassé, underfitting, sain, overfitting),
  avec une action pour chacune.
- Les bugs des training loops mystère identifiés à partir de leurs symptômes, et un bug que
  le test des 20 lignes **ne détecte pas**.
- Une AUC qui monte puis plafonne quand la capacité augmente.
- Un réseau à 5 couches sigmoïdes qui **perd** contre un réseau à 1 couche, et un
  réseau à 5 couches ReLU qui ne perd pas.
- La comparaison de la séance 9 reproduite : MLP 0,9124 contre HistGB 0,9216, cinq
  folds négatifs, p = 0,0062.

## Mode d'emploi

L'exercice 4.5 est le plus long à calculer (environ deux minutes) : lancez-le dès que
vous y arrivez, puis rédigez pendant qu'il tourne.


## Mise en route


In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / "src").is_dir() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import re
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from scipy import stats
from threadpoolctl import threadpool_limits
from torch import nn

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import GroupKFold, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from src.data import SEED, load_raw, set_seed, split_by_host
from src.training import EarlyStopping, evaluate, fit, loaders_from_arrays, make_mlp

set_seed(SEED)
torch.manual_seed(SEED)
# Quatre threads : plus rapide sur de petits réseaux, et identique aux conditions dans
# lesquelles les résultats de référence des séances 9 et 10 ont été obtenus (TD 3).
torch.set_num_threads(4)

df = load_raw()
train, _ = split_by_host(df)
train["licensed"] = train["license"].map(
    lambda s: bool(re.search(r"HUT|HB-|AJ0|ESFC", str(s))) if str(s) != "nan" else False
)
NUMERIC = [
    "accommodates",
    "bedrooms",
    "beds",
    "bathrooms",
    "latitude",
    "longitude",
    "minimum_nights",
    "number_of_reviews",
    "review_scores_rating",
    "calculated_host_listings_count",
    "availability_365",
]
CATEGORICAL = ["room_type", "property_type", "neighbourhood_cleansed"]
X = train[NUMERIC + CATEGORICAL]
y = train["licensed"].astype(int)
FOLDS = list(GroupKFold(n_splits=5).split(X, y, groups=train["host_id"]))


def make_preprocessor():
    return ColumnTransformer(
        [
            (
                "num",
                Pipeline(
                    [
                        ("impute", SimpleImputer(strategy="median")),
                        ("scale", StandardScaler()),
                    ]
                ),
                NUMERIC,
            ),
            (
                "cat",
                Pipeline(
                    [
                        ("impute", SimpleImputer(strategy="most_frequent")),
                        (
                            "onehot",
                            OneHotEncoder(
                                handle_unknown="ignore",
                                min_frequency=20,
                                sparse_output=False,
                            ),
                        ),
                    ]
                ),
                CATEGORICAL,
            ),
        ]
    )


train_idx, val_idx = FOLDS[0]
prep = make_preprocessor().fit(X.iloc[train_idx])
X_tr, X_va = prep.transform(X.iloc[train_idx]), prep.transform(X.iloc[val_idx])
y_tr, y_va = y.iloc[train_idx].to_numpy(), y.iloc[val_idx].to_numpy()
print(f"fold 0 : entraînement {X_tr.shape}, validation {X_va.shape}")

---

## Exercice 4.1 · Lire des courbes à l'aveugle

_Analyser · 12 min · ★★☆_

**Explication.** Trois allures canoniques, et un quatrième cas qui se déguise :

| allure                                               | diagnostic          | action                                                                                         |
| ---------------------------------------------------- | ------------------- | ---------------------------------------------------------------------------------------------- |
| les deux losses hautes, plates, ensemble             | **underfitting**    | plus de capacité, learning rate plus grand, plus d'epochs, _après_ avoir vérifié la machinerie |
| la training loss descend, la validation loss remonte | **overfitting**     | early stopping, régularisation (TD 5)                                                          |
| erratique, explosion, aucune tendance                | **cassé**           | déboguer (learning rate, scaling, formes) ; ne rien régler                                     |
| descente propre vers un plateau médiocre             | **l'un ou l'autre** | compter les unités vivantes avant de décider (TD 2)                                            |

**À vous.** Quatre entraînements réels, notés P, Q, R et S. Leurs réglages sont
cachés. Diagnostiquez-les **à partir des seules courbes**.


In [ ]:
courbes = pd.read_csv(
    ROOT / "assets" / "generated" / "cours_08_10_fr" / "courbes_mystere.csv"
)
fig, axes = plt.subplots(1, 4, figsize=(16, 3.6))
for ax, (nom, c) in zip(axes, courbes.groupby("run")):
    ax.plot(c["epoque"], c["train"], label="entraînement")
    ax.plot(c["epoque"], c["val"], label="validation")
    ax.set(title=f"entraînement {nom}", xlabel="epoch", ylabel="loss")
    ax.legend(fontsize=8)
plt.tight_layout()
plt.show()
print(
    courbes.groupby("run")
    .agg(
        train_debut=("train", "first"),
        train_fin=("train", "last"),
        val_min=("val", "min"),
        val_fin=("val", "last"),
    )
    .round(4)
)

In [ ]:
# TODO : pour chaque entraînement, un diagnostic et une action.
# diagnostic = {"P": "____", "Q": "____", "R": "____", "S": "____"}
# action = {"P": "____", "Q": "____", "R": "____", "S": "____"}

diagnostic = {
    "P": "cassé : loss initiale de plusieurs milliers, puis rien de lisible",
    "Q": "underfitting : deux losses hautes, qui descendent à peine et ensemble",
    "R": "sain : les deux losses descendent ; la validation se stabilise",
    "S": "overfitting : la training loss tombe vers zéro, la validation loss remonte",
}
action = {
    "P": "déboguer la machinerie : le learning rate (réduire d'un facteur 1 000), le scaling",
    "Q": "après vérification, plus de capacité et un learning rate plus grand",
    "R": "rien de fondamental ; early stopping pour garder la meilleure epoch",
    "S": "early stopping et régularisation (TD 5)",
}
# La révélation : les réglages cachés.
reglages = {
    "P": "64-32, learning rate 5,0 (Adam), 20 epochs",
    "Q": "2 unités, learning rate 1e-4, 20 epochs",
    "R": "64-32, learning rate 1e-3, 20 epochs",
    "S": "256-128-64 (61 185 paramètres), learning rate 1e-3, 40 epochs",
}
for nom in "PQRS":
    print(f"  {nom} : {reglages[nom]:61s} -> {diagnostic[nom].split(' :')[0]}")

**Analyse.** La lecture des courbes précède tout réglage. Deux pièges : confondre
**cassé** et **underfitting** (P n'a pas besoin de capacité, il a besoin d'un learning
rate mille fois plus petit), et oublier que la validation est la seule courbe qui compte
pour décider quand s'arrêter (S).

**Question 4.1.** Q et le réseau aux dead units (dying ReLU) du TD 2 ont des courbes
semblables.
Quelle mesure les distingue, et pourquoi leurs remèdes sont-ils opposés ?


---

## Exercice 4.2 · Cinq training loops mystère et le test des 20 lignes

_Résoudre · 15 min · ★★★_

**Explication.** Quand un réseau apprend mal, la checklist avant tout réglage :

1. **formes** : la sortie est-elle (n, 1) et la cible aussi ?
2. **données** : feature scaling, préprocesseur ajusté dans le fold, cible correcte ;
3. **loss** : des logits pour `BCEWithLogitsLoss`, pas des probabilités ;
4. **training loop** : `zero_grad`, `backward`, `step`, dans cet ordre ;
5. **learning rate** : essayer ×10 et ÷10 ;
6. **le test des 20 lignes** : un réseau correct doit pouvoir **mémoriser** vingt
   exemples jusqu'à une loss quasi nulle. S'il n'y arrive pas, la machinerie est
   cassée et aucun réglage ne le sauvera (Karpathy, 2019).

**À vous.** Voici cinq training loops sur les mêmes 20 lignes. Une seule est
correcte. Exécutez-les, lisez les symptômes, puis lisez le code pour trouver le bug
de chacune.


In [ ]:
idx20 = np.random.default_rng(0).choice(len(X_tr), 20, replace=False)
x20 = torch.tensor(X_tr[idx20], dtype=torch.float32)
y20 = torch.tensor(y_tr[idx20], dtype=torch.float32).reshape(-1, 1)


def nouveau_reseau():
    torch.manual_seed(0)
    return make_mlp(X_tr.shape[1], hidden=(64, 32))


def resultat(net, perte):
    with torch.no_grad():
        exact = ((torch.sigmoid(net(x20)) > 0.5).float() == y20).float().mean().item()
    return f"loss finale {perte:.4f}   accuracy sur les 20 lignes {exact:.2f}"


def mystere_1():
    net = nouveau_reseau()
    opt = torch.optim.Adam(net.parameters(), lr=1e-2)
    lf = nn.BCEWithLogitsLoss()
    for _ in range(300):
        opt.zero_grad()
        perte = lf(torch.sigmoid(net(x20)), y20)
        perte.backward()
        opt.step()
    return resultat(net, perte.item())


def mystere_2():
    net = nouveau_reseau()
    opt = torch.optim.Adam(net.parameters(), lr=1e-2)
    lf = nn.BCEWithLogitsLoss()
    for _ in range(300):
        opt.zero_grad()
        perte = lf(net(x20), y20)
        perte.backward()
    return resultat(net, perte.item())


def mystere_3():
    net = nouveau_reseau()
    opt = torch.optim.Adam(net.parameters(), lr=1e-2)
    for _ in range(300):
        opt.zero_grad()
        perte = ((torch.sigmoid(net(x20)) - y20.reshape(-1)) ** 2).mean()
        perte.backward()
        opt.step()
    return resultat(net, perte.item())


def mystere_4():
    net = nouveau_reseau()
    opt = torch.optim.Adam(net.parameters(), lr=1e-2)
    lf = nn.BCEWithLogitsLoss()
    for _ in range(300):
        perte = lf(net(x20), y20)
        perte.backward()
        opt.step()
    return resultat(net, perte.item())


def mystere_5():
    net = nouveau_reseau()
    opt = torch.optim.Adam(net.parameters(), lr=1e-2)
    lf = nn.BCEWithLogitsLoss()
    for _ in range(300):
        opt.zero_grad()
        perte = lf(net(x20), y20)
        perte.backward()
        opt.step()
    return resultat(net, perte.item())


print(f"  (sur ces 20 lignes, {int(y20.sum())} sont de classe 1)")
for k, f in enumerate([mystere_1, mystere_2, mystere_3, mystere_4, mystere_5], 1):
    print(f"  mystère {k} : {f()}")

In [ ]:
# TODO : pour chaque mystère, le bug (ou « aucun ») et le symptôme qui le trahit.
# bugs = {1: "____", 2: "____", 3: "____", 4: "____", 5: "____"}

bugs = {
    1: "double sigmoïde : la loss reçoit des probabilités, elle plafonne vers 0,48",
    2: "pas d'optimizer.step() : rien ne bouge, la loss reste à sa valeur initiale",
    3: "cible de forme (20,) contre sortie (20, 1) : le broadcasting calcule une loss "
    "sur 400 paires, le réseau ne mémorise pas",
    4: "pas de zero_grad : les gradients s'accumulent, MAIS Adam normalise le step et "
    "le réseau mémorise quand même : le test ne détecte pas ce bug",
    5: "aucun : loss quasi nulle, accuracy 1,00",
}
for k, texte in bugs.items():
    print(f"  mystère {k} : {texte}")

**Analyse.** Le test des 20 lignes coûte une seconde et détecte trois bugs sur
quatre. Le quatrième (pas de `zero_grad` avec Adam) passe le test : Adam divise chaque
step par une estimation de la taille du gradient, ce qui compense en partie
l'accumulation. Ce bug reste réel (avec SGD, ou sur des données plus grandes, il
dégrade l'entraînement, TD 2) : **un test réussi ne prouve pas l'absence de bug**, il
élimine une classe de bugs.

**Question 4.2.** Pourquoi le mystère 3 ne lève-t-il aucune erreur, alors que
`BCEWithLogitsLoss` en aurait levé une avec la même cible ?


---

## Exercice 4.3 · Capacité : plancher et plateau

_Expérimenter · 13 min · ★★☆_

**Explication.** La capacité d'un réseau grandit avec son nombre de paramètres. Pour
une couche dense de n entrées et m sorties : n·m poids + m biais. La séance 9 a mesuré
un plancher puis un plateau ; mesurez-le vous-même, avec trois seeds et l'early
stopping.


In [ ]:
# TODO : pour hidden dans [(2,), (16,), (64, 32), (256, 128, 64)] et les seeds 0, 1, 2 :
#   - loaders_from_arrays(..., batch_size=512, seed=graine), torch.manual_seed(graine),
#     make_mlp(77, hidden) ;
#   - fit(..., epochs=40, lr=1e-3, early_stopping=EarlyStopping(patience=6), seed=graine) ;
#   - AUC de validation.
# Tableau : paramètres, AUC moyenne, écart-type (ddof=1), epochs utilisées.
# capacite = None

lignes = []
for hidden in [(2,), (16,), (64, 32), (256, 128, 64)]:
    aucs, epoques = [], []
    for graine in (0, 1, 2):
        tr_l, va_l = loaders_from_arrays(
            X_tr, y_tr, X_va, y_va, batch_size=512, seed=graine
        )
        torch.manual_seed(graine)
        net = make_mlp(X_tr.shape[1], hidden=hidden)
        h = fit(
            net,
            tr_l,
            va_l,
            epochs=40,
            lr=1e-3,
            early_stopping=EarlyStopping(patience=6),
            seed=graine,
        )
        _, logits = evaluate(net, va_l, nn.BCEWithLogitsLoss())
        aucs.append(roc_auc_score(y_va, logits.ravel()))
        epoques.append(len(h["train"]))
    lignes.append(
        {
            "hidden layers": str(hidden),
            "paramètres": sum(p.numel() for p in net.parameters()),
            "AUC": round(float(np.mean(aucs)), 4),
            "écart-type": round(float(np.std(aucs, ddof=1)), 4),
            "epochs": epoques,
        }
    )
capacite = pd.DataFrame(lignes)
print(capacite.to_string(index=False))

In [ ]:
if capacite is not None:
    fig, ax = plt.subplots(figsize=(7, 3.6))
    ax.errorbar(
        capacite["paramètres"],
        capacite["AUC"],
        yerr=capacite["écart-type"],
        marker="o",
        capsize=4,
    )
    ax.set(
        xscale="log",
        xlabel="nombre de paramètres (échelle log)",
        ylabel="AUC de validation",
        title="Capacité : un plancher, puis un plateau",
    )
    plt.tight_layout()
    plt.show()

**Analyse.** En dessous d'un plancher, le réseau est en underfitting : quelques
centaines de paramètres ne suffisent pas. Au-dessus, l'AUC plafonne : des paramètres
en plus achètent une convergence plus rapide (moins d'epochs avant l'early stopping),
pas de l'AUC. **Il n'existe pas de troisième régime où « plus gros » veut
simplement dire « meilleur ».**

**Question 4.3.** Un réseau obtient une AUC basse et son early stopping ne se déclenche
jamais avant la fin du budget. Quels sont les deux problèmes compatibles avec ce
symptôme, et comment les départager ?


---

## Exercice 4.4 · Profondeur et vanishing gradient

_Comparer · 15 min · ★★☆_

**Explication.** Ajouter des couches ne sert que si le gradient **atteint** les
premières. Au TD 1, vous avez vu que la dérivée de la sigmoïde ne dépasse pas 0,25 :
mesurons ce que cela donne sur un vrai réseau.


In [ ]:
# TODO (a) : écrivez reseau_profond(n_couches, activation) = nn.Sequential de n_couches
# blocs [nn.Linear(., 64), activation()] suivis de nn.Linear(64, 1).
# Pour 8 hidden layers, sigmoïde puis ReLU (torch.manual_seed(0) avant chaque
# construction), calculez la loss sur les 512 premières lignes, appelez backward(), et
# affichez la norme du gradient des poids de chaque nn.Linear.


# def reseau_profond(n_couches, activation):
#     raise NotImplementedError


def reseau_profond(n_couches, activation):
    couches, largeur = [], X_tr.shape[1]
    for _ in range(n_couches):
        couches += [nn.Linear(largeur, 64), activation()]
        largeur = 64
    return nn.Sequential(*couches, nn.Linear(largeur, 1))


xb = torch.tensor(X_tr[:512], dtype=torch.float32)
yb = torch.tensor(y_tr[:512], dtype=torch.float32).reshape(-1, 1)
for nom, act in (("sigmoïde", nn.Sigmoid), ("ReLU", nn.ReLU)):
    torch.manual_seed(0)
    net = reseau_profond(8, act)
    nn.BCEWithLogitsLoss()(net(xb), yb).backward()
    normes = [m.weight.grad.norm().item() for m in net if isinstance(m, nn.Linear)]
    print(
        f"  {nom:9s} de la couche 1 à la couche 9 : "
        + "  ".join(f"{v:.0e}" for v in normes)
    )

**À vous : comparer.** Un réseau à 1 couche contre un réseau à 5 couches, avec trois
« machineries » différentes.


In [ ]:
# TODO (b) : entraînez 20 epochs (batches de 256, seed 0) les six combinaisons
#   profondeur 1 ou 5  x  {sigmoïde + SGD(lr=0.1), ReLU + SGD(lr=0.1), ReLU + Adam(lr=1e-3)}
# avec reseau_profond et fit(..., optimizer=...). Relevez l'AUC et la training loss
# finale.

MACHINERIES = {
    "sigmoïde + SGD": (nn.Sigmoid, lambda p: torch.optim.SGD(p, lr=0.1)),
    "ReLU + SGD": (nn.ReLU, lambda p: torch.optim.SGD(p, lr=0.1)),
    "ReLU + Adam": (nn.ReLU, lambda p: torch.optim.Adam(p, lr=1e-3)),
}
lignes = []
for nom, (act, fabrique) in MACHINERIES.items():
    for profondeur in (1, 5):
        tr_l, va_l = loaders_from_arrays(X_tr, y_tr, X_va, y_va, batch_size=256, seed=0)
        torch.manual_seed(0)
        net = reseau_profond(profondeur, act)
        h = fit(
            net, tr_l, va_l, epochs=20, optimizer=fabrique(net.parameters()), seed=0
        )
        _, logits = evaluate(net, va_l, nn.BCEWithLogitsLoss())
        lignes.append(
            {
                "machinerie": nom,
                "hidden layers": profondeur,
                "AUC": round(roc_auc_score(y_va, logits.ravel()), 4),
                "training loss finale": round(h["train"][-1], 4),
            }
        )
print(pd.DataFrame(lignes).to_string(index=False))

![Norme du gradient par couche](../assets/generated/cours_08_10_fr/f12_gradient_evanescent.png)

**Analyse.** Avec des sigmoïdes, le gradient qui atteint la première couche est des
millions de fois plus petit que celui de la dernière : les premières couches
n'apprennent presque rien, et **le réseau à 5 couches perd nettement contre le réseau
à 1 couche**. Sa training loss reste proche de celle du départ : il n'apprend
pas. Avec ReLU et SGD, le gradient passe, et 5 couches font mieux qu'une.

Avec ReLU et Adam, 5 couches perdent à nouveau, mais pour la raison **inverse** :
leur training loss est la plus basse du tableau. Le réseau profond apprend trop bien
le training set du fold et fait de l'overfitting (il n'y a pas d'early stopping ici).
Deux échecs de la profondeur, deux causes opposées, et c'est la training loss qui les
distingue. « Plus de couches, c'est mieux » est donc faux en
général : la profondeur ne paie que si le gradient arrive, et si la généralisation
est contrôlée (TD 5).

**Question 4.4.** Pourquoi les couches **proches des entrées** sont-elles les plus
touchées par le vanishing gradient ?


---

## Exercice 4.5 · La comparaison honnête

_Résoudre · 20 min · ★★★_

**Explication.** Le protocole des séances 6 et 7, appliqué au réseau :

1. **mêmes folds** (GroupKFold par hôte) et **même prétraitement** pour les deux
   modèles ;
2. le préprocesseur **réajusté dans chaque fold** ;
3. un **paired t-test** sur les différences par fold, et la lecture des **signes** ;
4. nommer les asymétries (ici, le MLP bénéficie de l'early stopping ; le booster tourne
   avec ses réglages par défaut).


In [ ]:
# TODO :
#  1. mlp_validation_croisee() : pour chaque fold k de FOLDS, préprocesseur ajusté sur la
#     partie entraînement, loaders_from_arrays(..., batch_size=256, seed=SEED),
#     torch.manual_seed(SEED + k), make_mlp(77, (64, 32)),
#     fit(..., epochs=60, lr=1e-3, early_stopping=EarlyStopping(patience=8), seed=SEED + k),
#     puis AUC sur la validation du fold.
#  2. Les AUC par fold de HistGradientBoostingClassifier(random_state=SEED) dans un
#     Pipeline avec make_preprocessor(), via cross_val_predict(..., cv=FOLDS).
#  3. Paired t-test (stats.ttest_rel), signes des différences, temps de calcul (limitez
#     HistGB à 4 threads avec threadpoolctl.threadpool_limits, comme torch).
#  4. Une phrase de conclusion pour le client.
# auc_mlp, auc_gb = None, None


def mlp_validation_croisee(
    hidden=(64, 32), epochs=60, lr=1e-3, batch_size=256, patience=8
):
    aucs, epoques = [], []
    for k, (tr, te) in enumerate(FOLDS):
        # Un préprocesseur neuf, ajusté DANS le fold : la ligne qu'une training loop
        # PyTorch écrite à la main oublie le plus souvent, faute de Pipeline pour
        # l'imposer.
        p = make_preprocessor().fit(X.iloc[tr])
        Xa, Xb = p.transform(X.iloc[tr]), p.transform(X.iloc[te])
        tr_l, va_l = loaders_from_arrays(
            Xa,
            y.iloc[tr].to_numpy(),
            Xb,
            y.iloc[te].to_numpy(),
            batch_size=batch_size,
            seed=SEED,
        )
        torch.manual_seed(SEED + k)
        net = make_mlp(Xa.shape[1], hidden=hidden)
        h = fit(
            net,
            tr_l,
            va_l,
            epochs=epochs,
            lr=lr,
            early_stopping=EarlyStopping(patience=patience),
            seed=SEED + k,
        )
        _, logits = evaluate(net, va_l, nn.BCEWithLogitsLoss())
        aucs.append(roc_auc_score(y.iloc[te], logits.ravel()))
        epoques.append(len(h["train"]))
    return np.array(aucs), epoques


debut = time.monotonic()
auc_mlp, epoques_mlp = mlp_validation_croisee()
duree_mlp = time.monotonic() - debut

debut = time.monotonic()
# Une comparaison de temps n'a de sens qu'à configuration égale : HistGB utilise par
# défaut tous les coeurs de la machine, alors que torch est limité à 4 threads. On le
# limite donc aussi à 4 threads (sur une machine à 20 threads, ses réglages par défaut
# le rendent plus de cinq fois plus lent, pour exactement la même AUC).
with threadpool_limits(limits=4):
    oof = cross_val_predict(
        Pipeline(
            [
                ("prep", make_preprocessor()),
                ("model", HistGradientBoostingClassifier(random_state=SEED)),
            ]
        ),
        X,
        y,
        cv=FOLDS,
        method="predict_proba",
    )[:, 1]
auc_gb = np.array([roc_auc_score(y.iloc[te], oof[te]) for _, te in FOLDS])
duree_gb = time.monotonic() - debut

diff = auc_mlp - auc_gb
test = stats.ttest_rel(auc_mlp, auc_gb)
print(
    f"  MLP    AUC = {auc_mlp.mean():.4f} +/- {auc_mlp.std():.4f}   {np.round(auc_mlp, 4)}"
)
print(
    f"  HistGB AUC = {auc_gb.mean():.4f} +/- {auc_gb.std():.4f}   {np.round(auc_gb, 4)}"
)
print(
    f"  MLP - HistGB : {diff.mean():+.4f}   signes {''.join('+' if d > 0 else '-' for d in diff)}"
    f"   paired t-test p = {test.pvalue:.4f}"
)
print(f"  epochs du MLP (early stopping) : {epoques_mlp}")
print(f"  temps de calcul : MLP {duree_mlp:.0f} s, HistGB {duree_gb:.0f} s")

In [ ]:
# Vérification : la séance 9 avait trouvé 0,9124 contre 0,9216 et p = 0,0062.
if auc_mlp is None:
    print("  à compléter")
else:
    ok = (
        abs(auc_mlp.mean() - 0.9124) < 5e-4
        and abs(auc_gb.mean() - 0.9216) < 5e-4
        and abs(test.pvalue - 0.0062) < 5e-4
    )
    print(
        f"  identique à la séance 9 : {'OK' if ok else 'différent (voir la remarque sur les threads, TD 3)'}"
    )

![Les cinq folds appariés](../assets/generated/cours_08_10_fr/f13_plis_apparies.png)

**À vous : conclure.** Écrivez la phrase destinée au client, puis la réponse à
l'objection d'un collègue.


In [ ]:
# TODO : deux textes courts.
# conclusion_client = "____"
# reponse_objection = "____"   # « Le booster a tourné par défaut, le MLP a eu l'early stopping. »


conclusion_client = (
    "Sur les annonces de Barcelone, avec des folds groupés par hôte, le réseau de neurones "
    "obtient une AUC de 0,9124 contre 0,9216 pour le gradient boosting ; il perd dans les "
    "cinq folds (paired t-test, p = 0,0062) et coûte plus de calcul à configuration égale. Nous recommandons le "
    "gradient boosting pour ce problème ; cette conclusion ne vaut que pour ces données "
    "tabulaires."
)
reponse_objection = (
    "L'asymétrie joue EN FAVEUR du réseau, qui perd malgré tout. Et la séance 9 a montré que "
    "le réseau est sur un plateau : 16 fois la batch size et 10 fois le learning rate ne "
    "déplacent l'AUC que de 0,0051 sur cinq folds, et 8,6 fois plus de paramètres ne "
    "l'améliorent pas."
)
print(conclusion_client, "\n")
print(reponse_objection)

**Analyse.** Le résultat n'est pas « le réseau a échoué » : c'est un **résultat négatif
établi rigoureusement**, avec le même protocole qui avait placé le booster au-dessus
de la logistic regression en séance 7. La conclusion est bornée : **sur ce
problème**, tabulaire, de taille moyenne, aux colonnes individuellement
significatives. Grinsztajn et al. (2022) retrouvent cette tendance sur 45 jeux de
données tabulaires.

**Question 4.5.** Donnez deux des trois raisons pour lesquelles les arbres gagnent
souvent sur les données tabulaires, et une situation où vous choisiriez quand même un
réseau.


---

## Synthèse

- **Lire les courbes avant de régler** : underfitting, overfitting, cassé, et le cas
  qui se déguise.
- La checklist, puis le **test des 20 lignes** : il détecte la plupart des bugs de
  machinerie, pas tous.
- **Capacité : un plancher, puis un plateau.** Au-dessus, plus de paramètres achètent
  de la vitesse, pas de l'AUC.
- **La profondeur ne paie que si le gradient arrive** : 5 couches sigmoïdes perdent
  contre 1 couche ; avec ReLU, non.
- **Le MLP perd contre le booster dans les cinq folds (p = 0,0062)** : un résultat
  établi, borné à ce problème.

## Questions de compréhension

1. Une training loss erratique appelle-t-elle plus de capacité ? Pourquoi ?
2. Que prouve un test des 20 lignes réussi ? Que ne prouve-t-il pas ?
3. Pourquoi plus de paramètres peuvent-ils réduire le nombre d'epochs sans améliorer
   l'AUC ?
4. Pourquoi le vanishing gradient touche-t-il surtout les premières couches ?
5. Énoncez le résultat MLP contre HistGB avec assez de précision pour qu'on puisse le
   vérifier.

## Pour aller plus loin

- Grinsztajn, L., Oyallon, E. et Varoquaux, G. (2022), _Why do tree-based models still
  outperform deep learning on tabular data?_, NeurIPS. https://arxiv.org/abs/2207.08815
- Shwartz-Ziv, R. et Armon, A. (2022), _Tabular Data: Deep Learning is Not All You
  Need_, Information Fusion 81. https://arxiv.org/abs/2106.03253
- Karpathy, A. (2019), _A Recipe for Training Neural Networks_ (dont « overfit a single
  batch »). http://karpathy.github.io/2019/04/25/recipe/
- Glorot, X. et Bengio, Y. (2010), _Understanding the difficulty of training deep
  feedforward neural networks_, AISTATS (vanishing gradient et saturation).

**Prochain TD :** le réseau fait de l'overfitting ; le régulariser, et prouver ce qui
marche.
